# Module 51 — Price Elasticity & Demand Curves

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Scikit-learn, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 41 (Marketing Mix Modeling)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Pricing Data](#3-synthetic-pricing-data)
4. [Demand Curve Estimation](#4-demand-curve-estimation)
5. [Price Elasticity](#5-price-elasticity)
6. [Revenue Optimization](#6-revenue-optimization)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why price elasticity matters?

Price elasticity measures **demand sensitivity to price changes**:
- **Pricing strategy**: Set optimal prices for revenue maximization.
- **Promotion planning**: Understand discount impact on demand.
- **Competitive positioning**: Price relative to competitors.

**Applications**:
1. **Dynamic pricing**: Adjust prices based on demand.
2. **Promotion optimization**: Optimize discount levels.
3. **Revenue management**: Maximize revenue per product.

**Key concepts**:
- **Elastic demand**: Price changes significantly affect demand.
- **Inelastic demand**: Price changes have little effect.
- **Optimal price**: Price that maximizes revenue.

In this notebook we will:
1. Generate synthetic pricing and demand data.
2. Estimate demand curves using regression.
3. Calculate price elasticity.
4. Find revenue-maximizing price.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Pricing Data

In [ ]:
# Generate synthetic pricing and demand data
N_OBSERVATIONS = 1000

# Price range
prices = np.random.uniform(10, 100, N_OBSERVATIONS)

# Demand model: Q = a - b*P + noise
true_intercept = 1000  # Base demand
true_slope = -8  # Price sensitivity

demand = true_intercept + true_slope * prices + np.random.normal(0, 50, N_OBSERVATIONS)
demand = np.maximum(demand, 0)  # Demand can't be negative

df = pd.DataFrame({
    'price': prices,
    'demand': demand,
    'revenue': prices * demand
})

print(f"✓ Generated {N_OBSERVATIONS} price-demand observations")
print(f"  Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")
print(f"  Demand range: {df['demand'].min():.0f} - {df['demand'].max():.0f}")
print(f"  Average revenue: ${df['revenue'].mean():,.2f}")

---
## §4 · Demand Curve Estimation

In [ ]:
# Estimate demand curve
X = df[['price']]
y = df['demand']

# Linear regression
model = LinearRegression()
model.fit(X, y)

# Predictions
price_range = np.linspace(10, 100, 100).reshape(-1, 1)
predicted_demand = model.predict(price_range)

print(f"✓ Demand curve estimated")
print(f"  Intercept (base demand): {model.intercept_:.2f}")
print(f"  Slope (price sensitivity): {model.coef_[0]:.2f}")
print(f"  R²: {model.score(X, y):.4f}")

---
## §5 · Price Elasticity

In [ ]:
# Calculate price elasticity
# Elasticity = (dQ/dP) * (P/Q)
def calculate_elasticity(price, demand, slope):
    """Calculate price elasticity at a given point."""
    return slope * (price / demand)

# Calculate elasticity at different price points
price_points = [20, 40, 60, 80]
elasticities = []

for price in price_points:
    demand = model.predict([[price]])[0]
    elasticity = calculate_elasticity(price, demand, model.coef_[0])
    elasticities.append({
        'price': price,
        'demand': demand,
        'elasticity': elasticity
    })

df_elasticity = pd.DataFrame(elasticities)

print(f"✓ Price elasticity calculated")
print(f"\nElasticity at different price points:")
print(df_elasticity.to_string(index=False))
print(f"\nInterpretation:")
print(f"  |E| > 1: Elastic (demand sensitive to price)")
print(f"  |E| < 1: Inelastic (demand insensitive to price)")
print(f"  |E| = 1: Unit elastic")

---
## §6 · Revenue Optimization

In [ ]:
# Find revenue-maximizing price
# Revenue = Price * Demand = P * (a - b*P) = aP - bP²
# dR/dP = a - 2bP = 0
# Optimal P = a / (2b)

optimal_price = -model.intercept_ / (2 * model.coef_[0])
optimal_demand = model.predict([[optimal_price]])[0]
optimal_revenue = optimal_price * optimal_demand

print(f"✓ Revenue optimization complete")
print(f"\nOptimal Price: ${optimal_price:.2f}")
print(f"Optimal Demand: {optimal_demand:.0f} units")
print(f"Maximum Revenue: ${optimal_revenue:,.2f}")

# Compare with current average price
avg_price = df['price'].mean()
avg_demand = model.predict([[avg_price]])[0]
avg_revenue = avg_price * avg_demand

print(f"\nComparison with average price (${avg_price:.2f}):")
print(f"  Revenue increase: ${optimal_revenue - avg_revenue:,.2f}")
print(f"  Percentage increase: {(optimal_revenue - avg_revenue) / avg_revenue * 100:.1f}%")

---
## §7 · Visualization

In [ ]:
# Visualize demand curve and revenue optimization
fig = make_subplots(rows=1, cols=2, subplot_titles=['Demand Curve', 'Revenue Curve'])

# Demand curve
fig.add_trace(go.Scatter(
    x=df['price'],
    y=df['demand'],
    mode='markers',
    name='Observed',
    marker=dict(size=5, opacity=0.5)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=price_range.flatten(),
    y=predicted_demand,
    mode='lines',
    name='Estimated',
    line=dict(color='red', width=3)
), row=1, col=1)

# Revenue curve
revenue_curve = price_range.flatten() * predicted_demand
fig.add_trace(go.Scatter(
    x=price_range.flatten(),
    y=revenue_curve,
    mode='lines',
    name='Revenue',
    line=dict(color='green', width=3)
), row=1, col=2)

# Mark optimal price
fig.add_vline(x=optimal_price, line_dash="dash", line_color="red", row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Demand decreases as price increases")
print("   - Revenue is maximized at optimal price")
print("   - Price elasticity varies along the demand curve")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Pricing Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Pricing strategy, promotion planning, revenue optimization |
> | **Method** | Log-log regression for elasticity estimation |
> | **Metrics** | Price elasticity, optimal price, revenue impact |
> | **Evaluation** | R², prediction accuracy, business validation |
> | **Deployment** | Dynamic pricing, A/B test price changes |
>
> ### 💡 Pricing Strategies
>
> | Elasticity | Strategy | Example |
> |------------|----------|--------|
> | Elastic (>1) | Lower prices, volume focus | Commodities |
> | Inelastic (<1) | Premium pricing, margin focus | Luxury goods |
> | Unit (=1) | Optimize mix | Most products |
>
> ### 🔑 Key Takeaways
>
> 1. **Price elasticity quantifies demand sensitivity** to price changes.
> 2. **Demand curves** show relationship between price and quantity.
> 3. **Optimal price** maximizes revenue (not necessarily highest price).
> 4. **Log-log regression** provides direct elasticity estimates.
> 5. **A/B test pricing changes** to validate elasticity estimates.